# baseline v3

이 베이스라인 코드는 `사전학습 모델 로드`, `배치 학습`, `파인튜닝`, `양자화`, `PEFT` 등이 적용된 버전입니다.

Colab의 GPU 환경에서 개발되었습니다.
- 런타임 - 런타임 유형 변경 - GPU로 변경(T4 GPU 등)



# 환경 준비

개발 환경에 필요한 라이브러리 버전을 고정하고 최신 버전으로 라이브러리를 업데이트합니다.

- 아래 셀 실행
- 실행 완료 후 런타임 - 세션 다시 시작

In [1]:
!pip -q install git+https://github.com/huggingface/transformers accelerate
!pip -q install --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio
!pip -q install "peft>=0.13.2" "bitsandbytes==0.46.1" datasets pillow pandas --upgrade

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import torch
print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())

Torch version: 2.10.0+cu128
CUDA version: 12.8
cuDNN version: 91002


# 데이터 준비

개발에 필요한 데이터를 준비합니다.

- train.csv, train 폴더
- test.csv, test 폴더
- sample_submission.csv

본 베이스라인은 colab에서 구글 드라이브를 마운트하여 사용합니다.

데이터를 압축 해제하는데 몇 분 정도의 시간이 소요됩니다.

#### 실습 참고 내용

    챕터 2-2 합성 데이터 실습
    - 구글 드라이브 마운트 : drive()

In [1]:
# 구글드라이브 마운트
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
#!gdown 1dt90DUU-Ktg-A3mcvRJfh5V-SC8ym6y4

In [11]:
# 압축 해제
#!unzip "2026-ssafy-ai-15-2.zip" -d "/content/"

# 라이브러리, 데이터, 설정

In [3]:
import os, re, math, random
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Dict, List, Any
from transformers import (
    Qwen3VLMoeForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm import tqdm

# 이미지 로드 시 픽셀 제한 해제
Image.MAX_IMAGE_PIXELS = None

# 디바이스 GPU 우선 사용 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# 사전 학습 모델 정의
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
MIN_IMAGE_SIZE = 256
MAX_IMAGE_SIZE = 1024
MAX_NEW_TOKENS = 8
SEED = 42
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# 데이터셋 로드
train_df = pd.read_csv("/content/train.csv")
test_df  = pd.read_csv("/content/test.csv")

# 학습데이터 200개만 추출
#train_df = train_df.sample(n=1200, random_state=SEED).reset_index(drop=True)

Device: cuda


# 모델, Processor

7.5GB 정도의 모델 다운로드가 진행됩니다. 10~20분 정도가 소요됩니다.

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - LoRA 구현 : LoraConfig()

In [4]:
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
# 양자화
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
# )

# 프로세서
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_IMAGE_SIZE*MIN_IMAGE_SIZE,
    max_pixels=MAX_IMAGE_SIZE*MAX_IMAGE_SIZE,
    trust_remote_code=True,
)

# 사전학습 모델
base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="sdpa"
)

# 양자화 모델로 로드
#base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()

# LoRA 세팅
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
)

# PEFT 모델 생성
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

trainable params: 87,293,952 || all params: 8,854,417,648 || trainable%: 0.9859


# 프롬프트 템플릿

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - 프롬프트 템플릿 : convert_to_chatml(), formatting_prompts_func()

In [5]:
# 모델 지시사항

SYSTEM_INSTRUCT = (
    "당신은 사물 인식 및 재활용 분류 전문가입니다. "
    "이미지에 포함된 물체의 외형, 질감, 브랜드 로고, 그리고 모든 텍스트(OCR) 정보를 정밀하게 분석하세요. "
    "질문에 대해 가장 적절한 선지를 선택하고, 반드시 소문자 'a', 'b', 'c', 'd' 중 한 글자만 답변하세요. "
    "설명이나 추가 텍스트는 절대로 포함하지 마세요."
)

# 프롬프트
def build_mc_prompt(question, a, b, c, d):
    return (
        f"### 지시사항:\n"
        f"이미지를 보고 아래 질문에 가장 적합한 답을 고르세요.\n\n"
        f"### 질문:\n"
        f"{question}\n\n"
        f"### 선택지:\n"
        f"(a) {a}\n"
        f"(b) {b}\n"
        f"(c) {c}\n"
        f"(d) {d}\n\n"
        f"### 주의:\n"
        f"1. 물체의 재질(금속, 유리, 종이, 플라스틱 등)과 표면 텍스트를 중점적으로 확인하세요.\n"
        f"2. 다른 설명 없이 오직 정답 알파벳 'a', 'b', 'c', 'd' 중 하나만 출력하세요."
    )

# Custom Dataset, Collator

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - TensorDataset()

    챕터 5-2 데이터 생성 및 파인튜닝 (향후 학습 분량)
    - IntentDataset()

In [6]:
# 커스텀 데이터셋
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")

        q = str(row["question"])
        a, b, c, d = str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
        user_text = build_mc_prompt(q, a, b, c, d)

        messages = [
            {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
            {"role":"user","content":[
                {"type":"image","image":img},
                {"type":"text","text":user_text}
            ]}
        ]
        if self.train:
            gold = str(row["answer"]).strip().lower()
            messages.append({"role":"assistant","content":[{"type":"text","text":gold}]})

        return {"messages": messages, "image": img}

# 데이터 콜레이터
@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts, images = [], []
        for sample in batch:
            messages = sample["messages"]
            img = sample["image"]

            text = self.processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )
            texts.append(text)
            images.append(img)

        enc = self.processor(
            text=texts,
            images=images,
            padding=True,
            return_tensors="pt"
        )

        if self.train:
            labels = enc["input_ids"].clone()
            pad_id = self.processor.tokenizer.pad_token_id

            for i in range(len(labels)):
                # 1. 패딩된 부분은 채점에서 제외 (-100)
                labels[i][labels[i] == pad_id] = -100

                # 2. 질문지(프롬프트) 부분 채점에서 제외 (-100)
                # 실제 데이터(패딩 제외)가 어디까지인지 길이 확인
                valid_len = (labels[i] != -100).sum()

                # 정답(a, b, c, d)은 항상 시퀀스의 맨 마지막에 생성됩니다.
                # 따라서 마지막 3개 토큰(정답, 종료토큰 등)만 채점 대상에 남기고,
                # 그 앞의 모든 질문지 부분은 -100으로 마스킹하여 무시합니다.
                labels[i][:valid_len - 3] = -100

            enc["labels"] = labels

        return enc


# DataLoader

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 데이터로더 정의 : DataLoader()

In [7]:
# 검증용 데이터 분리
split = int(len(train_df)*0.9)
train_subset, valid_subset = train_df.iloc[:split], train_df.iloc[split:]

# VQAMCDataset 형태로 변환
train_ds = VQAMCDataset(train_subset, processor, train=True)
valid_ds = VQAMCDataset(valid_subset, processor, train=True)

# 데이터로더
train_loader = DataLoader(train_ds, batch_size=3, shuffle=True, collate_fn=DataCollator(processor, True), num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=1, shuffle=False, collate_fn=DataCollator(processor, True), num_workers=0)

# fine-tuning

- 200개만 학습 : 10~20분 소요

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - 모델 정의 : SimpleMLP(), SequentialMLP()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [ ]:
from tqdm.auto import tqdm
import os
import shutil # 폴더 삭제를 위해 추가
import math
import torch
from transformers import get_cosine_with_hard_restarts_schedule_with_warmup

model = model.to(device)
GRAD_ACCUM = 8
EPOCHS = 60 # 원하시는 60 에폭으로 설정
LR = 5e-5

# 옵티마이저, 학습 스케줄러
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
num_training_steps = EPOCHS * math.ceil(len(train_loader)/GRAD_ACCUM)

# Cosine Annealing with Warm Restarts 설정
num_warmup_steps = int(num_training_steps * 0.1)
num_cycles = 1

scheduler = get_cosine_with_hard_restarts_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
    num_cycles=num_cycles
)

# 🌟 저장 로직을 위한 변수 초기화
best_val_accuracy = 0.0
previous_drive_save_dir = None # 드라이브의 이전 저장 경로만 추적 (삭제용)

# 학습 루프
global_step = 0
for epoch in range(EPOCHS):
    running_loss = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [train]", unit="batch")

    for step, batch in enumerate(progress_bar, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.amp.autocast('cuda', dtype=torch.bfloat16): # 최신 PyTorch 문법 적용
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        # 1. 역전파 추가
        loss.backward()

        # 원래의 loss 값을 누적
        running_loss += loss.item() * GRAD_ACCUM

        if step % GRAD_ACCUM == 0:
            # 2. 옵티마이저 스텝 추가
            optimizer.step()

            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

            # 3. 평균 Loss 출력
            avg_loss = running_loss / GRAD_ACCUM
            progress_bar.set_postfix({"loss": f"{avg_loss:.4f}"})

            running_loss = 0.0

    # -------------------------------------------------------------
    # 검증 (Validation)
    # -------------------------------------------------------------
    model.eval()
    val_loss = 0.0
    val_steps = 0
    correct_tokens = 0
    total_tokens = 0

    with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1} [valid]", unit="batch"):
            vb = {k:v.to(device) for k,v in vb.items()}
            outputs = model(**vb)

            val_loss += outputs.loss.item()
            val_steps += 1

            shift_logits = outputs.logits[..., :-1, :].contiguous()
            shift_labels = vb["labels"][..., 1:].contiguous()

            preds = torch.argmax(shift_logits, dim=-1)
            mask = shift_labels != -100

            correct_tokens += (preds[mask] == shift_labels[mask]).sum().item()
            total_tokens += mask.sum().item()

    val_accuracy = correct_tokens / total_tokens if total_tokens > 0 else 0.0
    print(f"[Epoch {epoch+1}] valid loss: {val_loss/val_steps:.4f} | valid accuracy: {val_accuracy:.4f}")

    model.train() # 다시 학습 모드로 전환

    # -------------------------------------------------------------
    # 🌟 이중 저장 로직 (Colab 세션 누적 + Google Drive 덮어쓰기)
    # -------------------------------------------------------------
    if val_accuracy > best_val_accuracy:
        print(f"🎉 최고 정확도 갱신! ({best_val_accuracy:.4f} -> {val_accuracy:.4f})")
        best_val_accuracy = val_accuracy

        # [1] Colab 로컬 세션에 저장 (이전 에폭 삭제 안 함, 누적 저장)
        LOCAL_SAVE_DIR = f"./session_best_models/epoch_{epoch+1}_acc_{best_val_accuracy:.4f}"
        os.makedirs(LOCAL_SAVE_DIR, exist_ok=True)
        model.save_pretrained(LOCAL_SAVE_DIR)
        processor.save_pretrained(LOCAL_SAVE_DIR)
        print(f"✅ [세션] 모델 저장 완료 (누적): {LOCAL_SAVE_DIR}")

        # [2] Google Drive에 저장 (최고 모델 1개만 유지)
        DRIVE_SAVE_DIR = f"/content/drive/MyDrive/Qwen_8B_Best_Model/epoch_{epoch+1}_acc_{best_val_accuracy:.4f}"
        os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
        model.save_pretrained(DRIVE_SAVE_DIR)
        processor.save_pretrained(DRIVE_SAVE_DIR)
        print(f"✅ [드라이브] 최고 모델 저장 완료: {DRIVE_SAVE_DIR}")

        # 드라이브의 이전 최고 모델 폴더가 있으면 삭제하여 용량 확보
        if previous_drive_save_dir is not None and os.path.exists(previous_drive_save_dir):
            shutil.rmtree(previous_drive_save_dir)
            print(f"🗑️ [드라이브] 이전 모델 삭제됨: {previous_drive_save_dir}")

        previous_drive_save_dir = DRIVE_SAVE_DIR # 드라이브 경로 업데이트
    else:
        print(f"아쉽게도 최고 정확도({best_val_accuracy:.4f})를 넘지 못했습니다.")

    print("-" * 50)

Epoch 1 [train]:   0%|          | 0/1522 [00:00<?, ?batch/s]

Epoch 1 [valid]:   0%|          | 0/508 [00:00<?, ?batch/s]

[Epoch 1] valid loss: 0.1003 | valid accuracy: 0.9587
🎉 최고 정확도 갱신! (0.0000 -> 0.9587)
✅ [세션] 모델 저장 완료 (누적): ./session_best_models/epoch_1_acc_0.9587
✅ [드라이브] 최고 모델 저장 완료: /content/drive/MyDrive/Qwen_8B_Best_Model/epoch_1_acc_0.9587
--------------------------------------------------


Epoch 2 [train]:   0%|          | 0/1522 [00:00<?, ?batch/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch 2 [valid]:   0%|          | 0/508 [00:00<?, ?batch/s]

[Epoch 2] valid loss: 0.0830 | valid accuracy: 0.9678
🎉 최고 정확도 갱신! (0.9587 -> 0.9678)
✅ [세션] 모델 저장 완료 (누적): ./session_best_models/epoch_2_acc_0.9678
✅ [드라이브] 최고 모델 저장 완료: /content/drive/MyDrive/Qwen_8B_Best_Model/epoch_2_acc_0.9678
🗑️ [드라이브] 이전 모델 삭제됨: /content/drive/MyDrive/Qwen_8B_Best_Model/epoch_1_acc_0.9587
--------------------------------------------------


Epoch 3 [train]:   0%|          | 0/1522 [00:00<?, ?batch/s]

# inference

30분~1시간 소요

#### 실습 참고 내용

    챕터4-1 RAG 기반 Customer Service AI 에이전트 개발
    - 데이터 파서 : langchain_core.output_parsers(), StrOutputParser()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [ ]:
# 데이터 파서 : 모델의 응답에서 선지를 추출
def extract_choice(text: str) -> str:
    text = text.strip().lower()

    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines:
        return "a"
    last = lines[-1]
    if last in ["a", "b", "c", "d"]:
        return last

    tokens = last.split()
    for tok in tokens:
        if tok in ["a", "b", "c", "d"]:
            return tok
    return "a"

# 추론을 위해 모든 레이어 활성화
model.eval()
preds = []

# 추론 루프
for i in tqdm(range(len(test_df)), desc="Inference", unit="sample"):
    row = test_df.iloc[i]
    img = Image.open(row["path"]).convert("RGB")
    user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])

    messages = [
        {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
        {"role":"user","content":[
            {"type":"image","image":img},
            {"type":"text","text":user_text}
        ]}
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img], return_tensors="pt").to(device)

    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=2, do_sample=False,
                                 eos_token_id=processor.tokenizer.eos_token_id)
    output_text = processor.batch_decode(out_ids, skip_special_tokens=True)[0]
    # print("output_text:", output_text)
    # print("extract_choice:", extract_choice(output_text))
    preds.append(extract_choice(output_text))

# 제출 파일 생성
submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv("submission2.csv", index=False)
print("Saved /content/submission.csv")

In [ ]:
# 모델 응답 예시
print(output_text)

In [ ]:
import requests

# 디스코드 채널 설정 -> 연동 -> 웹훅 만들기에서 복사한 URL
WEBHOOK_URL = "https://discord.com/api/webhooks/1489278333849043074/yEacNhKCSOKtonYxHs2dp4E7cH3Y1gvlHEoqbGo06CV8B9Cila-smE4vmFhh_3P2IeoM"

print("🚀 디스코드로 파일을 전송합니다...")
with open("submission2.csv", "rb") as f:
    response = requests.post(
        WEBHOOK_URL,
        files={"file": ("submission2.csv", f)},
        data={"content": "🎉 두도님! 자는 동안 모델 학습과 추론이 모두 완료되었습니다!"}
    )

if response.status_code in [200, 204]:
    print("✅ 디스코드 전송 완료!")
else:
    print(f"❌ 전송 실패: {response.status_code}")

In [ ]:
import os
from google.colab import userdata

# 1. 코랩 보안 비밀에서 캐글 인증 정보 불러오기
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

# 2. 캐글 라이브러리 최신화 (조용히 설치)
!pip install -q kaggle

# 3. 생성된 submission2.csv 파일을 해당 대회로 바로 제출
# -c: 대회 고유 이름 (URL에서 추출한 2026-ssafy-ai-15-2)
# -f: 제출할 파일명
# -m: 캐글 리더보드에 표시될 제출 메시지
!kaggle competitions submit -c 2026-ssafy-ai-15-2 -f submission2.csv -m "Auto submission before sleep"

print("✅ 캐글 제출 완료! 리더보드를 확인해 보세요.")